# Sesión 7: Pruebas de hipótesis y bootstrapping con la ENDI

En esta sesión aplicamos inferencia estadística sobre datos de la Encuesta Nacional de Desnutrición Infantil (ENDI). El objetivo es doble: entender los fundamentos de las pruebas de hipótesis y aprender a aplicarlas correctamente cuando los datos provienen de un diseño muestral complejo.

Trabajamos con tres variables de resultado:

| Variable | Tipo | Descripción |
|---|---|---|
| `dcronica` | Discreta (0/1) | Desnutrición crónica en menores de 5 años |
| `talla` | Continua | Talla consolidada en cm |

Y las cruzamos con: `area`, `sexo`, `etnia`, `region` y `prov`.

Cada ejercicio tiene una versión **sin diseño muestral** (usando `scipy`, como si la muestra fuera aleatoria simple) y una versión **con diseño muestral** (usando `svy`, que respeta estratos, conglomerados y factores de expansión). Comparar ambos resultados es parte del aprendizaje.

## Bloque 0: Preparación del entorno

Antes de ejecutar este notebook, asegúrate de activar el entorno virtual desde la carpeta raíz del repositorio.

```
cd <ruta/a carpeta/de trabajo>
.\venv\Scripts\activate
```

Si ya tenías una versión anterior del `requirements.txt` instalada, solo necesitas agregar el paquete nuevo:
```
pip install svy
```


Una vez activado, instala todas las dependencias con:
```
pip install -r requirements.txt
```

Si no tienes el archivo `requirements.txt`, instala los paquetes manualmente:
```
!pip install polars pyreadr scipy svy
```


### Ejercicio 0.1: Importar librerías

In [1]:
import polars as pl
import polars.selectors as cs
import pyreadr
import numpy as np
import scipy.stats as stats
import svy

### Ejercicio 0.2: Carga y selección de variables

Cargamos la tabla de personas desde el archivo `.rds` y seleccionamos únicamente las variables que necesitamos para esta sesión. La tabla original tiene más de 120 columnas; trabajar con un subconjunto acotado hace el análisis más claro y eficiente.

Las variables que seleccionamos son:

| Variable original | Nombre nuevo | Descripción |
|---|---|---|
| `id_upm` |  -  | Identificador de unidad primaria de muestreo |
| `estrato` |  -  | Estrato muestral |
| `fexp` |  -  | Factor de expansión normalizado |
| `area` |  -  | Área (1 = urbano, 2 = rural) |
| `region` |  -  | Región natural |
| `prov` |  -  | Provincia |
| `etnia` |  -  | Etnia del jefe de hogar |
| `f1_s1_2` | `sexo` | Sexo del niño o niña |
| `edaddias` |  -  | Edad en días |
| `f1_s5_5_1/2/3` | `long1/2/3` | Longitud (tres tomas, menores de 2 años) |
| `f1_s5_6_1/2/3` | `tal1/2/3` | Talla (tres tomas, 2 años y más) |
| `dcronica` |  -  | Desnutrición crónica oficial del INEC (0/1) |

In [ ]:
ruta_personas = r"..\..\Materiales\Insumos\ENDI\BDD_ENDI_R2_rds\BDD_ENDI_R2_f1_personas.rds"

personas = pl.from_pandas(pyreadr.read_r(ruta_personas)[None])

personas = personas.select([
    "id_upm", "estrato", "fexp",
    "area", "region", "prov", "etnia",
    "f1_s1_2","edaddias","f1_s5_5_1", "f1_s5_5_2", 
    "f1_s5_5_3","f1_s5_6_1", "f1_s5_6_2", "f1_s5_6_3",
    "dcronica",
]).rename({
    "f1_s1_2":   "sexo",
    "f1_s5_5_1": "long1", "f1_s5_5_2": "long2", "f1_s5_5_3": "long3",
    "f1_s5_6_1": "tal1",  "f1_s5_6_2": "tal2",  "f1_s5_6_3": "tal3",
})

personas.head(3)

id_upm,id_viv,id_hogar,id_per,id_mef,fecha_anio,fecha_mes,fecha_dia,fexp,estrato,area,region,prov,parr_pri,etnia,persona,altitud,edaddias,grupo_edad_nin,nivins_mef,f1_s1_1,f1_s1_2,f1_s1_3_1,f1_s1_4_1,f1_s1_4_2,f1_s1_4_3,f1_s1_5,f1_s1_6,f1_s1_6_2,f1_s1_7_a,f1_s1_7_b,f1_s1_7_c,f1_s1_7_d,f1_s1_7_e,f1_s1_7_f,f1_s1_8,f1_s1_9,…,f1_s5_3_2,f1_s5_3_3,f1_s5_4_1,f1_s5_4_2,f1_s5_4_3,f1_s5_5_1,f1_s5_5_2,f1_s5_5_3,f1_s5_6_1,f1_s5_6_2,f1_s5_6_3,f1_s5_7,f1_s6_1,f1_s6_2,f1_s6_3,f1_s6_4_1,f1_s6_4_2,f1_s6_5_1,f1_s6_5_2,f1_s6_5_3,f1_s6_6,quintil,pobreza,nbi_1,dcronica_2,dglobal_2,daguda_2,dcronica,dglobal,daguda,dcronica2_5,dglobal2_5,daguda2_5,ane6_59,ane6_59_new,ane6_23,ane6_23_new
str,str,str,str,str,str,str,str,f64,str,f64,f64,f64,f64,f64,str,i32,f64,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i64,i64,i64,i64,i64,i64,i64,i32,i64,…,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""0101500088""","""010150008801""","""01015000880101""","""0101500088010101""",null,"""2024""","""05""","""15""",0.297295,"""2712""",1.0,1.0,1.0,2.0,4.0,"""01""",2609,24473.0,null,null,1,1,67,14,5,1957,1,1,null,1,1,1,1,1,1,2,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""0101500088""","""010150008801""","""01015000880101""","""0101500088010102""",null,"""2024""","""05""","""15""",0.297295,"""2712""",1.0,1.0,1.0,2.0,4.0,"""02""",2609,23107.0,null,null,2,2,63,8,2,1961,1,1,null,1,1,1,1,1,1,2,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null
"""0101500088""","""010150008801""","""01015000880101""","""0101500088010103""",null,"""2024""","""05""","""15""",0.297295,"""2712""",1.0,1.0,1.0,2.0,4.0,"""03""",2609,14719.0,null,null,3,1,40,27,1,1984,1,1,null,1,1,1,1,1,1,2,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,3.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null


In [13]:
print(personas.shape) # pesos normalizados
print(personas["fexp"].sum()) # el total suma el tamaño de muestra. Se calculan proporciones y no totales

(93242, 16)
93241.99999999999


### Ejercicio 0.3: Filtro de población de análisis

Nos quedamos únicamente con los registros que cumplen dos condiciones:

1. El niño o niña tiene menos de 1826 días de edad (5 años exactos en días).
2. El indicador `dcronica` no es nulo, lo que implica que tiene una medición de talla válida y que el INEC pudo calcular el puntaje Z.

Además consolidamos la talla en una sola columna. La ENDI toma tres mediciones y registra longitud (menores de 2 años, acostados) o talla (mayores, de pie). La medida válida es el promedio de las dos tomas más cercanas entre sí. En esta sesión usamos una aproximación operativa: el promedio de las tomas no nulas disponibles en cada fila.

In [14]:
personas = (
    personas
    .filter((pl.col("edaddias") < 1826) & pl.col("dcronica").is_not_null())
    .with_columns(
        pl.when(pl.col("edaddias") < 730)
        .then(pl.mean_horizontal("long1", "long2", "long3"))
        .otherwise(pl.mean_horizontal("tal1", "tal2", "tal3"))
        .alias("talla")
    )
    .drop(["long1", "long2", "long3", "tal1", "tal2", "tal3"])
)

print(personas.shape)
personas.head(3)

(22331, 11)


id_upm,estrato,fexp,area,region,prov,etnia,sexo,edaddias,dcronica,talla
str,str,f64,f64,f64,f64,f64,i32,f64,f64,f64
"""0101500088""","""2712""",0.297295,1.0,1.0,1.0,4.0,1,1030.0,0.0,88.0
"""0101500088""","""2712""",0.297295,1.0,1.0,1.0,4.0,2,225.0,0.0,65.45
"""0101500088""","""2712""",0.297295,1.0,1.0,1.0,4.0,1,1611.0,0.0,102.1


### Ejercicio 0.4: Recodificación de etiquetas

Las variables categóricas vienen codificadas como números enteros. Las convertimos a etiquetas legibles para facilitar la interpretación de los resultados. Usamos `pl.when()` tal como en sesiones anteriores.

Los códigos de referencia son:

- **área**: 1 = urbano, 2 = rural  
- **sexo**: 1 = hombre, 2 = mujer  
- **región**: 1 = sierra, 2 = costa, 3 = amazonía  
- **etnia**: 1 = indígena, 2 = afroecuatoriano, 3 = montubio, 4 = mestizo, 5 = blanco, 6 = otro  
- **provincia**: casteamos a `String` porque es un código geográfico, no un número

In [15]:
personas = personas.with_columns([
    pl.when(pl.col("area") == 1).then(pl.lit("urbano"))
      .when(pl.col("area") == 2).then(pl.lit("rural"))
      .otherwise(pl.lit(None))
      .alias("area"),

    pl.when(pl.col("sexo") == 1).then(pl.lit("hombre"))
      .when(pl.col("sexo") == 2).then(pl.lit("mujer"))
      .otherwise(pl.lit(None))
      .alias("sexo"),

    pl.when(pl.col("region") == 1).then(pl.lit("sierra"))
      .when(pl.col("region") == 2).then(pl.lit("costa"))
      .when(pl.col("region") == 3).then(pl.lit("amazonia"))
      .otherwise(pl.lit(None))
      .alias("region"),

    pl.when(pl.col("etnia") == 1).then(pl.lit("indigena"))
      .when(pl.col("etnia") == 2).then(pl.lit("afroecuatoriano"))
      .when(pl.col("etnia") == 3).then(pl.lit("montubio"))
      .when(pl.col("etnia") == 4).then(pl.lit("mestizo"))
      .when(pl.col("etnia") == 5).then(pl.lit("blanco"))
      .when(pl.col("etnia") == 6).then(pl.lit("otro"))
      .otherwise(pl.lit(None))
      .alias("etnia"),

    pl.col("prov").cast(pl.Int8).cast(pl.String).str.pad_start(2,"0"),
])

personas.head(3)

id_upm,estrato,fexp,area,region,prov,etnia,sexo,edaddias,dcronica,talla
str,str,f64,str,str,str,str,str,f64,f64,f64
"""0101500088""","""2712""",0.297295,"""urbano""","""sierra""","""01""","""mestizo""","""hombre""",1030.0,0.0,88.0
"""0101500088""","""2712""",0.297295,"""urbano""","""sierra""","""01""","""mestizo""","""mujer""",225.0,0.0,65.45
"""0101500088""","""2712""",0.297295,"""urbano""","""sierra""","""01""","""mestizo""","""hombre""",1611.0,0.0,102.1


### Ejercicio 0.5: Declaración del diseño muestral

La ENDI usa un diseño **bietápico estratificado por conglomerados**:

- En la **primera etapa** se seleccionaron unidades primarias de muestreo (UPM) con probabilidad proporcional al tamaño dentro de cada estrato.
- En la **segunda etapa** se seleccionaron viviendas con niños menores de 5 años dentro de cada UPM seleccionada.

Esto significa que cada persona en la muestra **no tiene el mismo peso**: El factor de expansión (`fexp`) captura las diferencias.

Las tres variables que necesita `svy` para declarar el diseño son:

| Parámetro | Variable | Rol en el diseño |
|---|---|---|
| stratum | `estrato` | Define los estratos de muestreo |
| psu | `id_upm` | Identifica la unidad primaria de muestreo (conglomerado) |
| wgt | `fexp` | Factor de expansión normalizado (peso de cada observación) |

In [16]:
# Estratos reales y teóricos según el manual
print(personas.group_by("estrato").len().shape)
print(23*2*3+5*3) # se unen los estratos que no tienen suficientes UPM

(147, 2)
153


In [17]:
print(personas.shape) # pesos normalizados
print(personas["fexp"].sum()) # el total suma el tamaño de muestra. Se calculan proporciones y no totales

(22331, 11)
22240.02091028742


In [ ]:
personas

In [73]:
diseno = svy.Design(
    stratum="estrato",
    psu="id_upm",
    wgt="fexp",
)

muestra = svy.Sample(personas).set_design(diseno)

print(muestra)

  Survey Data
    Rows     : 22331
    Columns  : 12
    Strata   : 147
    PSUs     : 2820
  
  Survey Design
    Row index         : None
    Stratum           : estrato
    PSU               : id_upm
    SSU               : None
    Weight            : fexp
    With replacement  : False
    Prob              : None
    Hit               : None
    MOS               : None
    Population size   : None
    Replicate weights : None


## Bloque 1: Pruebas de hipótesis - fundamentos

Antes de escribir código, necesitamos entender qué estamos haciendo cuando realizamos una prueba de hipótesis.

### La lógica general

Una prueba de hipótesis responde a una pregunta del tipo: *¿es posible que lo que observo en la muestra sea simplemente producto del azar, o hay evidencia de que existe una diferencia real en la población?*

Para responder esa pregunta construimos dos hipótesis:

- **$H_0$ (hipótesis nula):** no hay diferencia. Lo que observamos es ruido aleatorio.
- **$H_1$ (hipótesis alternativa):** sí hay diferencia. Existe un efecto real.

Luego calculamos un **estadístico de test** que resume cuán alejados están los datos de lo que esperaríamos si H₀ fuera verdadera. A partir de ese estadístico obtenemos el **p-valor**: la probabilidad de observar un resultado tan extremo como el nuestro *suponiendo que H₀ es cierta*.

- Si el p-valor es muy pequeño (por convención, menor a 0.05), rechazamos H₀.
- Si es grande, no tenemos evidencia suficiente para rechazarla.

### Los dos tipos de error

| -- | H₀ verdadera | H₀ falsa |
|---|---|---|
| **Rechazamos H₀** | Error tipo I (falso positivo) | Decisión correcta |
| **No rechazamos H₀** | Decisión correcta | Error tipo II (falso negativo) |

El nivel de significancia α = 0.05 es exactamente el límite que fijamos para el error tipo I: aceptamos equivocarnos en un 5% de los casos cuando H₀ es verdadera.

### Por qué el diseño muestral importa

Las pruebas clásicas de `scipy` asumen que cada observación fue seleccionada con la **misma probabilidad** y de forma **independiente** (muestreo aleatorio simple). Cuando los datos provienen de un diseño complejo como la ENDI:

- Las personas dentro de una misma UPM se parecen más entre sí que personas de distintas UPM (correlación intraclúster).
- Las probabilidades de selección varían entre estratos y áreas.

Ignorar esto **subestima los errores estándar**, lo que hace que las pruebas sean demasiado optimistas: encontramos diferencias significativas que en realidad no lo son. `svy` corrige este problema usando la **linealización de Taylor** para estimar la varianza correcta bajo el diseño declarado.

## Bloque 2: Prueba de normalidad

Antes de decidir entre una prueba paramétrica (que asume distribución normal) y una no paramétrica, verificamos si la variable continua que queremos analizar se distribuye aproximadamente como una normal.

Usamos dos pruebas complementarias:

**Shapiro-Wilk** es potente para muestras pequeñas (n < 50). El estadístico W mide qué tan bien se ajustan los datos a una línea recta en un gráfico Q-Q. W cercano a 1 indica normalidad.

$$W = \frac{\left(\sum_{i=1}^{n} a_i x_{(i)}\right)^2}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

- **$a_i$** está precalculado y depende del n del vector.
- **$H_0$:** los datos provienen de una distribución normal.
- **$H_1$:** los datos no provienen de una distribución normal.

**Kolmogorov-Smirnov** compara la distribución empírica con la distribución normal teórica de misma media y desviación estándar. Es más adecuado para muestras grandes como la nuestra.

$$D = \sup_x |F_n(x) - F(x)|$$

donde $F_n(x)$ es la distribución empírica y $F(x)$ es la normal teórica.

- **$H_0$:** los datos provienen de una distribución normal.
- **$H_1$:** los datos no provienen de una distribución normal.

### Ejercicio 2.1: Normalidad de la talla

Aplicamos Shapiro-Wilk sobre una submuestra de 50 observaciones (el test es poco fiable con muestras muy grandes porque casi cualquier desviación mínima resulta significativa) y Kolmogorov-Smirnov sobre la muestra completa.

In [74]:
talla = personas["talla"].drop_nulls().to_numpy()

np.random.seed(123) # la semilla fija los números pseudoaleatorios

# Shapiro-Wilk sobre una submuestra
# submuestra = np.random.choice(talla, size=50, replace=False)
# stat_sw, p_sw = stats.shapiro(submuestra)

stat_sw, p_sw = stats.shapiro(talla)
print(f"Shapiro-Wilk (n=50): W = {stat_sw:.4f}, p = {p_sw:.4f}")

# Kolmogorov-Smirnov sobre la muestra completa
stat_ks, p_ks = stats.kstest(talla, "norm",args=(talla.mean(), talla.std()))
print(f"Kolmogorov-Smirnov (n={len(talla)}): D = {stat_ks:.4f}, p = {p_ks:.4f}")

Shapiro-Wilk (n=50): W = 0.9766, p = 0.0000
Kolmogorov-Smirnov (n=22329): D = 0.0638, p = 0.0000


c:\Users\fdjp1\Documents\Clases\FLCS_ESEDA\venv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 22329.
  res = hypotest_fun_out(*samples, **kwds)


### Conclusión

Con p = 0.0231 en Shapiro-Wilk y p < 0.0001 en Kolmogorov-Smirnov, rechazamos $H_0$ en ambas pruebas: la talla **no se distribuye normalmente** en la muestra.


## Bloque 3: Chi-cuadrado - asociación entre desnutrición crónica y área

La prueba de chi-cuadrado evalúa si existe asociación entre dos variables categóricas. En nuestro caso: ¿la prevalencia de desnutrición crónica es diferente en el área urbana respecto a la rural?

- **$H_0$:** la proporción de desnutrición crónica es igual en ambas áreas (son independientes).
- **$H_1$:** existe asociación entre el área y la desnutrición crónica.

El estadístico de prueba compara las frecuencias observadas con las que esperaríamos si no hubiera asociación:

$$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

donde $O_{ij}$ son las frecuencias observadas y $E_{ij} = \frac{(\text{total fila}_i)(\text{total columna}_j)}{n}$ son las frecuencias esperadas bajo independencia.

Con diseño muestral complejo, `svy` estima proporciones por grupo con su error estándar correcto. La comparación equivalente se hace con un **test de Wald** sobre la diferencia de proporciones.

### Ejercicio 3.1: Chi-cuadrado sin diseño muestral

Construimos la tabla de contingencia con recuentos simples (sin ponderar) y aplicamos la prueba de chi-cuadrado de `scipy`.

In [75]:
# Tabla de contingencia sin ponderar
tabla = (
    personas
    .filter(pl.col("area").is_not_null() & pl.col("dcronica").is_not_null())
    .group_by(["area", "dcronica"])
    .len()
    .sort(["area", "dcronica"])
    .pivot(on="dcronica", index="area", values="len")
)
print("Tabla de contingencia (sin ponderar):")
print(tabla)
print("")

# Chi-cuadrado
matriz = tabla.drop("area").to_numpy()
chi2, p_chi2, gl, esperados = stats.chi2_contingency(matriz)

print(f"Chi2 = {chi2:.4f}")
print(f"Grados de libertad = {gl}")
print(f"p-valor = {p_chi2:.4f}")
print("")
print("Frecuencias esperadas bajo H0:")
print(pl.DataFrame(esperados))

Tabla de contingencia (sin ponderar):
shape: (2, 3)
┌────────┬───────┬──────┐
│ area   ┆ 0.0   ┆ 1.0  │
│ ---    ┆ ---   ┆ ---  │
│ str    ┆ u32   ┆ u32  │
╞════════╪═══════╪══════╡
│ rural  ┆ 6727  ┆ 2027 │
│ urbano ┆ 11611 ┆ 1966 │
└────────┴───────┴──────┘

Chi2 = 272.1704
Grados de libertad = 1
p-valor = 0.0000

Frecuencias esperadas bajo H0:
shape: (2, 2)
┌──────────────┬─────────────┐
│ column_0     ┆ column_1    │
│ ---          ┆ ---         │
│ f64          ┆ f64         │
╞══════════════╪═════════════╡
│ 7188.699655  ┆ 1565.300345 │
│ 11149.300345 ┆ 2427.699655 │
└──────────────┴─────────────┘


### Conclusión

Con $\chi^2$ = 272.17, 1 grado de libertad y p < 0.0001, rechazamos $H_0$: existe asociación estadísticamente significativa entre el área (urbano/rural) y la desnutrición crónica. La prevalencia de desnutrición crónica **no es igual** en ambas áreas.

Ahora bien, esta prueba se calculó **sin incorporar el diseño muestral**, tratando cada observación como si tuviera el mismo peso. En el ejercicio 3.2 veremos si esta conclusión se mantiene cuando corregimos por estratificación, conglomeración y factores de expansión.

### Ejercicio 3.2: Proporciones con diseño muestral

Con `svy` estimamos la proporción de desnutrición crónica por área incorporando el diseño muestral. El resultado incluye el error estándar correcto bajo estratificación y conglomeración.

La comparación formal entre grupos se hace con un **test de Wald**:

$$t = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\widehat{SE}(\hat{p}_1)^2 + \widehat{SE}(\hat{p}_2)^2}}$$

Los grados de libertad se aproximan como el número de estratos menos uno.

In [76]:
# Proporciones con diseño muestral
est_area = muestra.estimation.prop("dcronica", by="area")
print("Proporciones de dcronica por área (con diseño):")
print(est_area)
print("")

# Filtrar estimaciones para dcronica == 1 por área
props = {
    r.by_level[0]: {"est": r.est, "se": r.se} for r in est_area.estimates if r.y_level == 1.0
}

areas = list(props.keys())
est1, se1 = props[areas[0]]["est"], props[areas[0]]["se"]
est2, se2 = props[areas[1]]["est"], props[areas[1]]["se"]

# Test de Wald
diff = est1 - est2
se_diff = np.sqrt(se1**2 + se2**2)
t_wald = diff / se_diff
gl = est_area.n_strata - 1
p_wald = 2 * stats.t.sf(abs(t_wald), df=gl)

print(f"Diferencia ({areas[0]} - {areas[1]}): {diff:.4f}")
print(f"t de Wald = {t_wald:.4f}")
print(f"p-valor (con diseño) = {p_wald:.4f}")

Proporciones de dcronica por área (con diseño):
Estimate: PROP (TAYLOR)

  area    dcronica     est      se     lci     uci  cv (%)
  ------  --------  ------  ------  ------  ------  ------
  rural   0         0.7881  0.0074  0.7733  0.8022  0.94  
  rural   1         0.2119  0.0074  0.1978  0.2267  3.49  
  urbano  0         0.8463  0.0055  0.8353  0.8567  0.64  
  urbano  1         0.1537  0.0055  0.1433  0.1647  3.55  

Diferencia (urbano - rural): -0.0582
t de Wald = -6.3390
p-valor (con diseño) = 0.0000


### Conclusión

Con t de Wald = 6.34 y p < 0.0001, rechazamos $H_0$: existe una diferencia estadísticamente significativa en la prevalencia de desnutrición crónica entre áreas.



## Bloque 4: T-test - diferencia de medias de talla por sexo

Queremos saber si la talla media de los niños difiere de la de las niñas.

### T-test de dos muestras independientes

Asume que las muestras provienen de poblaciones normales con varianzas posiblemente distintas (versión de Welch).

- **$H_0$:** $\mu_{\text{hombre}} = \mu_{\text{mujer}}$
- **$H_1$:** $\mu_{\text{hombre}} \neq \mu_{\text{mujer}}$

$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$


### Ejercicio 4.1: T-test sin diseño muestral

In [90]:
hombres = personas.filter(pl.col("sexo") == "hombre")["talla"].drop_nulls().to_numpy()
mujeres = personas.filter(pl.col("sexo") == "mujer")["talla"].drop_nulls().to_numpy()

t_stat, p_ttest = stats.ttest_ind(hombres, mujeres, equal_var=False)

print(f"Media niños: {hombres.mean():.4f} cm")
print(f"Media niñas: {mujeres.mean():.4f} cm")
print(f"Diferencia:    {hombres.mean() - mujeres.mean():.4f} cm")
print()
print(f"t = {t_stat:.4f}")
print(f"p-valor (sin diseño) = {p_ttest:.4f}")

Media niños: 87.0124 cm
Media niñas: 85.7387 cm
Diferencia:    1.2737 cm

t = 7.0134
p-valor (sin diseño) = 0.0000


### Ejercicio 4.2: T-test con diseño muestral

Con diseño muestral estimamos la media de talla por sexo usando linealización de Taylor, y construimos el estadístico de Wald:

$$t_{\text{diseño}} = \frac{\hat{\mu}_1 - \hat{\mu}_2}{\sqrt{\widehat{SE}(\hat{\mu}_1)^2 + \widehat{SE}(\hat{\mu}_2)^2}}$$

In [91]:
est_sexo = muestra.estimation.mean("talla", by="sexo", drop_nulls=True)
print("Media de talla por sexo (con diseño):")
print(est_sexo)
print()

# Extraer estimaciones
e = {r.by_level[0]: r for r in est_sexo.estimates}
est_h, se_h = e["hombre"].est, e["hombre"].se
est_m, se_m = e["mujer"].est,  e["mujer"].se

# Estadístico de Wald
diff = est_h - est_m
se_diff = np.sqrt(se_h**2 + se_m**2)
t_diseno = diff / se_diff
gl = est_sexo.n_strata - 1
p_diseno = 2 * stats.t.sf(abs(t_diseno), df=gl)

print(f"Diferencia (niños - niñas): {diff:.4f} cm")
print(f"SE de la diferencia: {se_diff:.4f}")
print(f"t de Wald = {t_diseno:.4f}")
print(f"Grados de libertad = {gl}")
print(f"p-valor (con diseño) = {p_diseno:.4f}")

Media de talla por sexo (con diseño):
Estimate: MEAN (TAYLOR)

  sexo        est      se      lci      uci  cv (%)
  ------  -------  ------  -------  -------  ------
  hombre  86.9999  0.2161  86.5762  87.4236  0.25  
  mujer   85.3907  0.2264  84.9469  85.8346  0.27  

Diferencia (niños - niñas): 1.6091 cm
SE de la diferencia: 0.3129
t de Wald = 5.1419
Grados de libertad = 146
p-valor (con diseño) = 0.0000


## Bloque 5: Bootstrapping

El bootstrapping es una técnica de remuestreo que permite estimar la variabilidad de cualquier estadístico sin asumir una distribución teórica. La idea es sencilla:

1. Toma la muestra original de tamaño $n$.
2. Extrae con reemplazo $n$ observaciones de esa muestra. Esto es una **réplica bootstrap**.
3. Calcula el estadístico de interés en esa réplica.
4. Repite el proceso $B$ veces.
5. La distribución de los $B$ estadísticos aproxima la distribución muestral real.

La varianza bootstrap del estadístico $\hat{\theta}$ se estima como:

$$\widehat{Var}_{boot}(\hat{\theta}) = \frac{1}{B-1} \sum_{b=1}^{B} \left(\hat{\theta}^{(b)} - \bar{\hat{\theta}}\right)^2$$

donde $\hat{\theta}^{(b)}$ es el estadístico calculado en la réplica $b$ y $\bar{\hat{\theta}}$ es el promedio de las $B$ réplicas.

Lo usamos para estimar la varianza de la **prevalencia nacional de desnutrición crónica** y verificar que el resultado es consistente con la varianza que calcula `svy` mediante linealización de Taylor.

### Ejercicio 5.1: Varianza de la prevalencia nacional con diseño muestral

`svy` estima la varianza usando linealización de Taylor, que es el método oficial del INEC para la ENDI. El error estándar al cuadrado es la varianza.

In [85]:
est_nacional = muestra.estimation.prop("dcronica")
print("Prevalencia nacional de desnutrición crónica (con diseño):")
print(est_nacional)
print()

# Extraer varianza para dcronica == 1
e_nac = [r for r in est_nacional.estimates if r.y_level == 1.0][0]
var_svy = e_nac.se ** 2

print(f"Prevalencia:          {e_nac.est:.6f}")
print(f"Error estándar (svy): {e_nac.se:.6f}")
print(f"Varianza (svy):       {var_svy:.8f}")

Prevalencia nacional de desnutrición crónica (con diseño):
Estimate: PROP (TAYLOR)

  dcronica     est      se     lci     uci  cv (%)
  --------  ------  ------  ------  ------  ------
  0         0.8253  0.0044  0.8165  0.8338  0.54  
  1         0.1747  0.0044  0.1662  0.1835  2.53  

Prevalencia:          0.174690
Error estándar (svy): 0.004418
Varianza (svy):       0.00001952


### Ejercicio 5.2: Replicar la varianza con bootstrap manual

Remuestreamos las UPM dentro de cada estrato (no observaciones individuales) para respetar la estructura del diseño muestral. Este es el bootstrap apropiado para muestras complejas: en lugar de sortear personas, sorteamos conglomerados.

In [98]:
np.random.seed(123)
B = 1000

# Preparar datos con id de UPM, estrato, peso y variable de interés
datos_boot = personas.select(["id_upm", "estrato", "fexp", "dcronica"]).drop_nulls()

estratos = datos_boot["estrato"].unique().to_list()
prev_boot = np.empty(B)

for b in range(B):
    replicas = []
    for estrato in estratos:
        upms_estrato = (
            datos_boot
            .filter(pl.col("estrato") == estrato)["id_upm"]
            .unique()
            .to_list()
        )
        # Remuestrear UPM con reemplazo dentro del estrato
        upms_boot = np.random.choice(upms_estrato, size=len(upms_estrato), replace=True)
        for upm in upms_boot:
            replicas.append(datos_boot.filter(pl.col("id_upm") == upm))
    replica_df = pl.concat(replicas)
    # Prevalencia ponderada en esta réplica
    num = (replica_df["dcronica"] * replica_df["fexp"]).sum()
    den = replica_df["fexp"].sum()
    prev_boot[b] = num / den

var_boot = prev_boot.var(ddof=1)
se_boot  = np.sqrt(var_boot)

print(f"Varianza bootstrap:       {var_boot:.8f}")
print(f"Error estándar bootstrap: {se_boot:.6f}")
print()
print(f"Varianza svy (Taylor):    {var_svy:.8f}")
print(f"Error estándar svy:       {e_nac.se:.6f}")
print()
print(f"Diferencia relativa: {abs(var_boot - var_svy) / var_svy * 100:.2f}%")

Varianza bootstrap:       0.00001757
Error estándar bootstrap: 0.004192

Varianza svy (Taylor):    0.00001952
Error estándar svy:       0.004418

Diferencia relativa: 9.98%
